# Modelling: Classical

This notebook cleans and splits the data into training, validation, and test sets, then builds, tunes, and compares two classical models: Random Forest and XGBoost. Both train on `CORE_TRAITS`, the 10-feature behavioural set `03_eda.ipynb` justified (it excludes `Characteristics` and every other raw header field, removing the DLL-vs-EXE shortcut at the source).

Training uses `data/dataset_malwares.csv` directly, the original published dataset, with no data added or removed to influence results. An earlier version of this pipeline added real legitimate files to the training data specifically to reduce false positives found during real-world testing (see `07_real_world_validation.ipynb`'s note on this); that approach was removed because changing training data to fix a specific known weakness is not sound data science practice, it risks overfitting to one evaluation target rather than producing a model that genuinely generalises. Any real-world false-positive rate measured in `07_real_world_validation.ipynb` against this unaugmented model should be reported honestly as a finding, not treated as something to engineer away by reshaping the input data.

Cleaning is done step by step below (matching `02_data_preparation.ipynb`), rather than through a shared function, so the effect of each step is visible on its own.

The test set is not touched in this notebook. Final model selection between these two and the MLP from `05_modelling_mlp.ipynb`, plus the one-time test-set check, happens in `06_evaluation.ipynb`.

In [1]:
# Standard library
import sys

# Third-party
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
sys.path.append("../src")

from constants import CORE_TRAITS, ID_COLUMNS, LABEL_COLUMN, RANDOM_STATE

## Load the Raw Dataset

In [2]:
df = pd.read_csv("../data/dataset_malwares.csv")
print(df.shape)

(19611, 79)


## Select CORE_TRAITS and the Label

Scoped to `CORE_TRAITS` (10 features) rather than all 77, matching the columns this notebook actually trains on. `02_data_preparation.ipynb` cleans on all 77 features instead, for a general cleaned dataset, so the duplicate count here is expected to differ from that notebook's.

In [3]:
X = df[CORE_TRAITS].copy()
y = df[LABEL_COLUMN].copy()
print(f"X: {X.shape}, y: {y.shape}")

X: (19611, 10), y: (19611,)


## Drop Rows with Missing Values

In [4]:
before = len(X)
mask = X.notna().all(axis=1)
X, y = X[mask], y[mask]
dropped_na = before - len(X)
print(f"Dropped {dropped_na} rows with missing values. {len(X)} rows remain.")

Dropped 0 rows with missing values. 19611 rows remain.


## Drop Duplicate Feature Rows

Scoping cleaning to just the 10 `CORE_TRAITS` columns makes exact duplicates more likely than `02_data_preparation.ipynb`'s all-77-feature cleaning found (2,984 there), since fewer columns need to match for two rows to count as identical.

In [5]:
before = len(X)
dup_mask = ~X.duplicated()
X, y = X[dup_mask], y[dup_mask]
dropped_dup = before - len(X)
print(f"Dropped {dropped_dup} duplicate feature rows. {len(X)} rows remain.")

Dropped 6094 duplicate feature rows. 13517 rows remain.


## Check the Class Balance

In [6]:
y.value_counts(normalize=True) * 100

Malware
1    64.245025
0    35.754975
Name: proportion, dtype: float64

64.25% malicious / 35.76% benign after this `CORE_TRAITS`-scoped cleaning, close to `02_data_preparation.ipynb`'s all-77-feature result (70.16%/29.84%) but not identical, since a different, larger set of rows was dropped as duplicates here.

## Split into Train, Validation, and Test

70% train, 15% validation, 15% test, stratified so each split keeps the same class balance. `RANDOM_STATE` is fixed so `05_modelling_mlp.ipynb` and `06_evaluation.ipynb` can reproduce this exact split without saving it to disk.

In [7]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE)

for name, split_y in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(f"{name}: {len(split_y)} rows, {split_y.mean()*100:.2f}% malicious")

Train: 9461 rows, 64.24% malicious
Validation: 2028 rows, 64.25% malicious
Test: 2028 rows, 64.25% malicious


## Build the Model Pipelines

Each pipeline bundles a `StandardScaler` with its classifier, so the exact same fitted scaler is used at train and predict time. `scale_pos_weight` is computed fresh from the actual training split (not hardcoded), since cleaning changes the effective class balance.

In [8]:
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"Training class balance: {neg} benign / {pos} malicious -> XGBoost scale_pos_weight={scale_pos_weight:.4f}")

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)),
])
xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # RF gets class balancing for free via class_weight="balanced", XGBoost needs scale_pos_weight explicitly
    ("model", XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight)),
])

Training class balance: 3383 benign / 6078 malicious -> XGBoost scale_pos_weight=0.5566


## Tune and Evaluate Random Forest

`GridSearchCV` scores on macro F1, not accuracy, so both classes count equally despite the imbalance. `n_estimators` = number of trees, `max_depth` = how deep each can grow, `cv=5` tests each combination on 5 different folds and averages them.

In [9]:
rf_grid = GridSearchCV(
    rf_pipeline,
    param_grid={"model__n_estimators": [200, 400], "model__max_depth": [12, 24, None]},
    scoring="f1_macro", cv=5, n_jobs=-1,
)
rf_grid.fit(X_train, y_train)
rf_best = rf_grid.best_estimator_
print("Random Forest best params:", rf_grid.best_params_)

Random Forest best params: {'model__max_depth': None, 'model__n_estimators': 400}


In [10]:
rf_val_preds = rf_best.predict(X_val)
print(classification_report(y_val, rf_val_preds, target_names=["benign", "malicious"]))
print("Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):")
print(confusion_matrix(y_val, rf_val_preds))
print("ROC-AUC:", round(roc_auc_score(y_val, rf_best.predict_proba(X_val)[:, 1]), 4))

              precision    recall  f1-score   support

      benign       0.90      0.93      0.91       725
   malicious       0.96      0.94      0.95      1303

    accuracy                           0.94      2028
   macro avg       0.93      0.93      0.93      2028
weighted avg       0.94      0.94      0.94      2028

Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):
[[ 672   53]
 [  76 1227]]
ROC-AUC: 0.9824


## Tune and Evaluate XGBoost

Same tuning approach, plus `learning_rate` (how much each extra tree corrects the previous ones), the same grid `train_classical.py` uses.

In [11]:
xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid={
        "model__n_estimators": [200, 400],
        "model__max_depth": [4, 8],
        "model__learning_rate": [0.05, 0.1],
    },
    scoring="f1_macro", cv=5, n_jobs=-1,
)
xgb_grid.fit(X_train, y_train)
xgb_best = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)

XGBoost best params: {'model__learning_rate': 0.1, 'model__max_depth': 8, 'model__n_estimators': 200}


In [12]:
xgb_val_preds = xgb_best.predict(X_val)
print(classification_report(y_val, xgb_val_preds, target_names=["benign", "malicious"]))
print("Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):")
print(confusion_matrix(y_val, xgb_val_preds))
print("ROC-AUC:", round(roc_auc_score(y_val, xgb_best.predict_proba(X_val)[:, 1]), 4))

              precision    recall  f1-score   support

      benign       0.90      0.94      0.92       725
   malicious       0.97      0.94      0.95      1303

    accuracy                           0.94      2028
   macro avg       0.93      0.94      0.94      2028
weighted avg       0.94      0.94      0.94      2028

Confusion matrix (rows=actual, cols=predicted, order=[benign, malicious]):
[[ 683   42]
 [  76 1227]]
ROC-AUC: 0.9832


## Summary

- Loaded the raw `dataset_malwares.csv` and cleaned it step by step, scoped to `CORE_TRAITS`: 0 rows dropped for missing values, 6,094 dropped as duplicate-feature rows, 13,517 remain, class balance 64.25% malicious / 35.76% benign, all confirmed via real, executed output above.
- Splits the cleaned data 70/15/15 into train/validation/test, stratified, using a fixed `RANDOM_STATE` so `05_modelling_mlp.ipynb` and `06_evaluation.ipynb` reproduce the exact same rows.
- Tunes Random Forest and XGBoost with 5-fold `GridSearchCV` scored on macro F1, over the same parameter grids `train_classical.py` originally used.
- **TODO (needs a fresh Restart & Run All on a machine with scikit-learn/xgboost installed):** the split, pipeline, and tuning cells above need to be re-run before their output, best hyperparameters, and validation metrics can be trusted. Fill in the actual accuracy, F1, and ROC-AUC numbers here once that re-run is done.
- The test set was not touched. `06_evaluation.ipynb` rebuilds both candidates alongside the MLP from `05_modelling_mlp.ipynb`, selects a winner, and runs the one-time test-set check.